First, we download OGGMs projections, e.g. like this:
wget -c -nc -r -np -nH --cut-dirs=7 -A "*.nc" https://cluster.klima.uni-bremen.de/~oggm/oggm-standard-projections/oggm_v16/2025.6/era5/per_glacier_spinup/CMIP6/2100/RGI01/

In [1]:
OGGM_data_folder = "oggm_data/2100/RGI01"

We inspect one of OGGMs batch output files.

In [2]:
import xarray as xr
example_file_name = "run_hydro_era5_gcm_merged_MRI-ESM2-0_ssp119_bc_2000_2019_endyr2101_Batch_0_1000.nc"
ds = xr.load_dataset(OGGM_data_folder + "/" + example_file_name)
ds

<xarray.Dataset> Size: 47MB
Dimensions:                       (time: 102, rgi_id: 1000, month_2d: 12)
Coordinates:
  * time                          (time) float64 816B 2e+03 ... 2.101e+03
    hydro_year                    (time) int64 816B 2000 2001 2002 ... 2100 2101
    hydro_month                   (time) int64 816B 4 4 4 4 4 4 ... 4 4 4 4 4 4
    calendar_year                 (time) int64 816B 2000 2001 2002 ... 2100 2101
    calendar_month                (time) int64 816B 1 1 1 1 1 1 ... 1 1 1 1 1 1
  * rgi_id                        (rgi_id) <U14 56kB 'RGI60-01.00001' ... 'RG...
  * month_2d                      (month_2d) int64 96B 1 2 3 4 5 ... 9 10 11 12
    calendar_month_2d             (month_2d) int64 96B 1 2 3 4 5 ... 9 10 11 12
Data variables: (12/29)
    volume                        (time, rgi_id) float32 408kB 6.713e+06 ... ...
    volume_bsl                    (time, rgi_id) float32 408kB 0.0 0.0 ... 0.0
    volume_bwl                    (time, rgi_id) float32 408kB 0.0 0.0 ... 0.0
    area                          (time, rgi_id) float32 408kB 3.553e+05 ... ...
    length                        (time, rgi_id) float32 408kB 828.0 ... 760.0
    calving                       (time, rgi_id) float32 408kB 0.0 0.0 ... 0.0
    ...                            ...
    snowfall_on_glacier_monthly   (time, month_2d, rgi_id) float32 5MB 6.153e...
    snow_bucket_monthly           (time, month_2d, rgi_id) float32 5MB 0.0 .....
    residual_mb_monthly           (time, month_2d, rgi_id) float32 5MB 0.0 .....
    water_level                   (rgi_id) float32 4kB 0.0 0.0 0.0 ... 0.0 0.0
    glen_a                        (rgi_id) float32 4kB 8.025e-24 ... 8.025e-24
    fs                            (rgi_id) float32 4kB 0.0 0.0 0.0 ... 0.0 0.0
Attributes:
    description:    OGGM model output
    oggm_version:   1.6.3.dev30+gbba6e5f24
    calendar:       365-day no leap
    creation_date:  2025-12-17 21:27:43

We find data variables per glacier over time (either annual or monthly) and flow calibration parameters per glacier (I wonder why the MB calibration parameters are not included here?).

Lets first create the regional GlacierMIP output files:
We start with loading all the batch nc-files from one region and one simulation setup. For this example, we assume that our OGGM data folder only contains these nc-files.

In [3]:
from pathlib import Path
files = sorted(Path(OGGM_data_folder).glob("*.nc"))

ds_mf = xr.open_mfdataset(
    files,
    combine="by_coords",
)

ds_mf

<xarray.Dataset> Size: 1GB
Dimensions:                       (time: 102, rgi_id: 27108, month_2d: 12)
Coordinates:
  * time                          (time) float64 816B 2e+03 ... 2.101e+03
    hydro_year                    (time) int64 816B dask.array<chunksize=(102,), meta=np.ndarray>
    hydro_month                   (time) int64 816B dask.array<chunksize=(102,), meta=np.ndarray>
    calendar_year                 (time) int64 816B dask.array<chunksize=(102,), meta=np.ndarray>
    calendar_month                (time) int64 816B dask.array<chunksize=(102,), meta=np.ndarray>
  * rgi_id                        (rgi_id) <U14 2MB 'RGI60-01.00001' ... 'RGI...
  * month_2d                      (month_2d) int64 96B 1 2 3 4 5 ... 9 10 11 12
    calendar_month_2d             (month_2d) int64 96B dask.array<chunksize=(12,), meta=np.ndarray>
Data variables: (12/29)
    volume                        (time, rgi_id) float32 11MB dask.array<chunksize=(102, 1000), meta=np.ndarray>
    volume_bsl                    (time, rgi_id) float32 11MB dask.array<chunksize=(102, 1000), meta=np.ndarray>
    volume_bwl                    (time, rgi_id) float32 11MB dask.array<chunksize=(102, 1000), meta=np.ndarray>
    area                          (time, rgi_id) float32 11MB dask.array<chunksize=(102, 1000), meta=np.ndarray>
    length                        (time, rgi_id) float32 11MB dask.array<chunksize=(102, 1000), meta=np.ndarray>
    calving                       (time, rgi_id) float32 11MB dask.array<chunksize=(102, 1000), meta=np.ndarray>
    ...                            ...
    snowfall_on_glacier_monthly   (time, month_2d, rgi_id) float32 133MB dask.array<chunksize=(102, 12, 1000), meta=np.ndarray>
    snow_bucket_monthly           (time, month_2d, rgi_id) float32 133MB dask.array<chunksize=(102, 12, 1000), meta=np.ndarray>
    residual_mb_monthly           (time, month_2d, rgi_id) float32 133MB dask.array<chunksize=(102, 12, 1000), meta=np.ndarray>
    water_level                   (rgi_id) float32 108kB dask.array<chunksize=(1000,), meta=np.ndarray>
    glen_a                        (rgi_id) float32 108kB dask.array<chunksize=(1000,), meta=np.ndarray>
    fs                            (rgi_id) float32 108kB dask.array<chunksize=(1000,), meta=np.ndarray>
Attributes:
    description:    OGGM model output
    oggm_version:   1.6.3.dev30+gbba6e5f24
    calendar:       365-day no leap
    creation_date:  2025-12-17 21:27:43

For a regional aggregation we need to sum the values of individual glaciers per data variable.

In [4]:
regional_ds = ds_mf.sum(dim="rgi_id")
regional_ds

<xarray.Dataset> Size: 51kB
Dimensions:                       (time: 102, month_2d: 12)
Coordinates:
  * time                          (time) float64 816B 2e+03 ... 2.101e+03
    hydro_year                    (time) int64 816B dask.array<chunksize=(102,), meta=np.ndarray>
    hydro_month                   (time) int64 816B dask.array<chunksize=(102,), meta=np.ndarray>
    calendar_year                 (time) int64 816B dask.array<chunksize=(102,), meta=np.ndarray>
    calendar_month                (time) int64 816B dask.array<chunksize=(102,), meta=np.ndarray>
  * month_2d                      (month_2d) int64 96B 1 2 3 4 5 ... 9 10 11 12
    calendar_month_2d             (month_2d) int64 96B dask.array<chunksize=(12,), meta=np.ndarray>
Data variables: (12/29)
    volume                        (time) float32 408B dask.array<chunksize=(102,), meta=np.ndarray>
    volume_bsl                    (time) float32 408B dask.array<chunksize=(102,), meta=np.ndarray>
    volume_bwl                    (time) float32 408B dask.array<chunksize=(102,), meta=np.ndarray>
    area                          (time) float32 408B dask.array<chunksize=(102,), meta=np.ndarray>
    length                        (time) float32 408B dask.array<chunksize=(102,), meta=np.ndarray>
    calving                       (time) float32 408B dask.array<chunksize=(102,), meta=np.ndarray>
    ...                            ...
    snowfall_on_glacier_monthly   (time, month_2d) float32 5kB dask.array<chunksize=(102, 12), meta=np.ndarray>
    snow_bucket_monthly           (time, month_2d) float32 5kB dask.array<chunksize=(102, 12), meta=np.ndarray>
    residual_mb_monthly           (time, month_2d) float32 5kB dask.array<chunksize=(102, 12), meta=np.ndarray>
    water_level                   float32 4B dask.array<chunksize=(), meta=np.ndarray>
    glen_a                        float32 4B dask.array<chunksize=(), meta=np.ndarray>
    fs                            float32 4B dask.array<chunksize=(), meta=np.ndarray>
Attributes:
    description:    OGGM model output
    oggm_version:   1.6.3.dev30+gbba6e5f24
    calendar:       365-day no leap
    creation_date:  2025-12-17 21:27:43

Now we are in excellent conditions for converting to the GlacierMIP4 standardized output.
First we create an empty GlacierMIP4 template file.

In [5]:
import yaml
from src.timeseries import get_days_and_bounds
from src.dataset_builder import create_template_nc, add_variable

frequency = "annual"

EPOCH_DATE = "1850-01-01"
START_DATE = "2000-01-01"
END_DATE = "2101-01-02"

epoch, time, bounds = get_days_and_bounds(
    start_date=START_DATE,
    end_date=END_DATE,
    epoch_date=EPOCH_DATE,
    frequency=frequency,
)

glaciermip_ds = create_template_nc(
    time=time,
    bounds=bounds,
    frequency=frequency,
    epoch=epoch,
)

with open("variables.yml", "r") as f:
    VARS = yaml.safe_load(f)["variables"]

for var_key, meta in VARS.items():

    glaciermip_ds = add_variable(
        glaciermip_ds,
        var_key,
        meta=meta,
        time_dim="annual_time",
    )

glaciermip_ds

<xarray.Dataset> Size: 3kB
Dimensions:             (annual_time: 102, nbounds: 2)
Coordinates:
  * annual_time         (annual_time) int64 816B 54786 55152 ... 91311 91676
  * nbounds             (nbounds) int64 16B 0 1
Data variables:
    annual_time_bounds  (annual_time, nbounds) int32 816B 54421 54786 ... 91676
    area                (annual_time) float32 408B 0.0 0.0 0.0 ... 0.0 0.0 0.0
    mass                (annual_time) float32 408B 0.0 0.0 0.0 ... 0.0 0.0 0.0
    mass_bsl            (annual_time) float32 408B 0.0 0.0 0.0 ... 0.0 0.0 0.0
    frontal_abl         (annual_time) float32 408B 0.0 0.0 0.0 ... 0.0 0.0 0.0

Then we write the data from OGGM's regional ds into the standardized GlacierMIP ds. 

In [6]:
# Let's add mass first in OGGM's data
GLACIERMIP_ICE_DENSITY = 900  # kg m-3
regional_ds["mass"] = regional_ds["volume"] * GLACIERMIP_ICE_DENSITY
regional_ds["mass_bsl"] = regional_ds["volume_bsl"] * GLACIERMIP_ICE_DENSITY

# We need a lookup table for the variable names (could be stored in external file)
glaciermip_names_to_oggm_names = {"area":"area", "frontal_abl":"calving", "mass":"mass", "mass_bsl":"mass_bsl"}

# Then we iterate over the data variables and simply paste the data
# Of course user's have to make sure that the times and units are correct in the original data!
for var_name, da in glaciermip_ds.data_vars.items():
    # Exlude time bounds
    if "bounds" in var_name:
        continue
    
    oggm_name = glaciermip_names_to_oggm_names[var_name]
    glaciermip_ds[var_name].values = regional_ds[oggm_name].values

glaciermip_ds

<xarray.Dataset> Size: 3kB
Dimensions:             (annual_time: 102, nbounds: 2)
Coordinates:
  * annual_time         (annual_time) int64 816B 54786 55152 ... 91311 91676
  * nbounds             (nbounds) int64 16B 0 1
Data variables:
    annual_time_bounds  (annual_time, nbounds) int32 816B 54421 54786 ... 91676
    area                (annual_time) float32 408B 8.796e+10 ... 6.335e+10
    mass                (annual_time) float32 408B 1.777e+16 ... 1.098e+16
    mass_bsl            (annual_time) float32 408B 8.603e+14 ... 3.936e+14
    frontal_abl         (annual_time) float32 408B 0.0 0.0 0.0 ... 0.0 0.0 0.0

That's it, we can store the dataset with the correct naming.

In [7]:
GLACIER_MODEL = "OGGM"
RGI_REGION = "01"
GCM = "MRI-ESM2-0"
SSP = "119"

filename = GLACIER_MODEL + "_" + "rgi" + RGI_REGION + "_" + GCM + "_" + SSP + "_annual.nc"

# toDo: What compression should be used?
glaciermip_ds.to_netcdf(filename)


In [8]:
from ncplot import view

glaciermip_ds = glaciermip_ds.set_index(annual_time="annual_time")

glaciermip_ds = glaciermip_ds.drop_dims("nbounds")
view(glaciermip_ds, "mass")


/home/johannes/mambaforge/envs/glaciermip-nc/lib/python3.12/site-packages/ncplot/plot.py:15: UserWarning: Unable to import cartopy. For better plots install cartopy or check cartopy installation!
  warnings.warn(


BokehModel(combine_events=True, render_bundle={'docs_json': {'e7c509e3-6d7e-4f24-829d-00c840f70b47': {'version…